In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
books = pd.read_csv("complete_data.csv")
books

,Title,description,authors,categories,image,previewLink,unique_values,tagged_description
0,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,Philip Nel,Biography & Autobiography,http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,1,1 Philip Nel takes a fascinating look into the...
1,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,David R. Ray,Religion,http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,2,2 This resource includes twelve principles in ...
2,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,Veronica Haddon,Fiction,http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,3,3 Julia Thomas finds her life spinning out of ...
3,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,Everett Ferguson,Religion,http://books.google.com/books/content?id=kVqRa...,http://books.google.nl/books?id=kVqRaiPlx88C&p...,4,4 In The Church of Christ: A Biblical Ecclesio...
4,Saint Hyacinth of Poland,The story for children 10 and up of St. Hyacin...,Mary Fabyan Windeatt,Biography & Autobiography,http://books.google.com/books/content?id=lmLqA...,http://books.google.nl/books?id=lmLqAAAACAAJ&d...,5,5 The story for children 10 and up of St. Hyac...
...,...,...,...,...,...,...,...,...
128103,Final things,Grace's father believes in science and builds ...,Jenny Offill,Fiction,http://books.google.com/books/content?id=UbSFB...,http://books.google.com/books?id=UbSFBAAAQBAJ&...,128104,128104 Grace's father believes in science and ...
128104,The Magic of the Soul: Applying Spiritual Powe...,"""The Magic of the Soul, Applying Spiritual Pow...",Patrick J. Harbula,"Body, Mind & Spirit",http://books.google.com/books/content?id=H1ELA...,http://books.google.com/books?id=H1ELAAAACAAJ&...,128105,"128105 ""The Magic of the Soul, Applying Spirit..."
128105,Autodesk Inventor 10 Essentials Plus,Autodesk Inventor 2017 Essentials Plus provide...,"Daniel Banach, Travis Jones",Computers,http://books.google.com/books/content?id=zxHRC...,http://books.google.com/books?id=zxHRCwAAQBAJ&...,128106,128106 Autodesk Inventor 2017 Essentials Plus ...
128106,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",Elvira Woodruff,Juvenile Fiction,http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...,128107,"128107 During a school trip to Ellis Island, D..."


In [4]:
books["tagged_description"].to_csv("description.txt",
                            index = False,
                            header = False)

In [4]:
from langchain_core.documents import Document
raw_documents = TextLoader("description.txt", encoding="utf-8").load()
documents = []
text = raw_documents[0].page_content

for line in text.split("\n"):
    line = line.strip()
    if line:
        documents.append(Document(page_content=line))


In [5]:
documents[80945]

Document(metadata={}, page_content='"80946 For courses in Certification, and Microsoft Certification. The text is intended to help students study, train, and prepare for specific computer, networking, and system installation certification examinations related to the MCSE exams sponsored by Microsoft, as well as software related products by Microsoft. It is crafted and intended for students in an academic setting, with appropriate pedagogical tools and practice problems for MCSA and MCSE."')

In [16]:
db_books = Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings(),
    persist_directory="chroma_store"
)

db_books.persist()

In [3]:
db_books = Chroma(
    persist_directory="chroma_store",
    embedding_function=OpenAIEmbeddings()
)

In [5]:
query = "magical adventure"
docs = db_books.similarity_search(query, k=10)
docs

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [22]:
value = int(docs[0].page_content.split()[0].strip().replace('"', ''))

books[books["unique_values"] == value]


,Title,description,authors,categories,image,previewLink,unique_values,tagged_description
68725,Catseye (Shattered Light),"A magical world of swords, sorcery, and advent...","William R. Forstchen, Jaki Demarest",Fiction,http://books.google.com/books/content?id=aS4GA...,http://books.google.nl/books?id=aS4GAAAACAAJ&d...,68726,"68726 A magical world of swords, sorcery, and ..."


In [23]:
def retrieve_recommendations(query: str, top_k: int = 10,) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 10)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.split()[0].strip().replace('"', ''))]

    return books[books["unique_values"].isin(books_list)]

In [24]:
retrieve_recommendations("science fiction about time travel")

,Title,description,authors,categories,image,previewLink,unique_values,tagged_description
37082,Time ship,There is a secret passage through time ...and ...,Stephen Baxter,Fiction,http://books.google.com/books/content?id=iTtsn...,http://books.google.com/books?id=iTtsn_JYTAgC&...,37083,37083 There is a secret passage through time ....
45518,The Science in Science Fiction,An illustrated survey of the actual science be...,"Peter Nicholls, David Langford, Brian M. Stabl...",Science,http://books.google.com/books/content?id=LRb8X...,http://books.google.com/books?id=LRb8X7Ust_oC&...,45519,45519 An illustrated survey of the actual scie...
47729,Cronos,"In a trio of novels--Letters from Atlantis, Pr...",Robert Silverberg,Fiction,http://books.google.com/books/content?id=dp8JA...,http://books.google.com/books?id=dp8JAAAACAAJ&...,47730,47730 In a trio of novels--Letters from Atlant...
54159,Breaking the Time Barrier: The Race to Build t...,Provides a close-up look at the cutting-edge r...,Jenny Randles,History,http://books.google.com/books/content?id=mnOmA...,http://books.google.com/books?id=mnOmAwAAQBAJ&...,54160,54160 Provides a close-up look at the cutting-...
94343,Cryptozoic,A novel of time-traveling adventure from the a...,Brian W. Aldiss,Fiction,http://books.google.com/books/content?id=ERMeA...,http://books.google.com/books?id=ERMeAwAAQBAJ&...,94344,94344 A novel of time-traveling adventure from...
99011,Bent Man,This classic work of science fiction is widely...,David Gerrold,Fiction,http://books.google.com/books/content?id=Tl0VB...,http://books.google.com/books?id=Tl0VBQAAQBAJ&...,99012,99012 This classic work of science fiction is ...
101283,Einstein's Universe,A Princeton astrophysicist explores whether jo...,J. Richard Gott,Science,http://books.google.com/books/content?id=3QBgC...,http://books.google.com/books?id=3QBgCgAAQBAJ&...,101284,101284 A Princeton astrophysicist explores whe...
116455,The End of Eternity,Story of a special group of technicians in the...,Isaac Asimov,Fiction,http://books.google.com/books/content?id=XHJcA...,http://books.google.com/books?id=XHJcAAAAMAAJ&...,116456,116456 Story of a special group of technicians...
118697,Time: a Traveler's Guide,"With the aid of diagrams, a science-fiction ta...",Clifford A. Pickover,Science,http://books.google.com/books/content?id=luD1B...,http://books.google.com/books?id=luD1Bpc1fmsC&...,118698,"118698 With the aid of diagrams, a science-fic..."
126996,The time machine (A Watermill classic),Illus. in black-and-white. When a turn-of-the-...,H. G. Wells,Fiction,http://books.google.com/books/content?id=wVrAi...,http://books.google.com/books?id=wVrAiSJffcAC&...,126997,126997 Illus. in black-and-white. When a turn-...
